# ANCHOR 

Este notebook:
- Carga un **modelo preentrenado de torchvision** (ResNet50).
- Aplica **Anchors** para imágenes con **Alibi `AnchorImage`** (superpíxeles).
- Procesa **todas** las imágenes de una carpeta y guarda resultados en `output_anchor/`:
  - PNG con: original, **anchor masked image**, superpíxeles, y overlay 
  - TXT con: clase predicha, probabilidad, **precision** y **coverage**.


In [ ]:
# (Opcional) Instala dependencias si no las tienes
# !pip install -U torch torchvision pillow matplotlib numpy
# !pip install -U alibi scikit-image

import os
from pathlib import Path
import numpy as np
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")      # o "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")       # Apple Silicon
else:
    device = torch.device("cpu")

print("Device:", device)


Usando: mps
Device: mps


In [3]:
# Imports de Alibi (AnchorImage)

try:
    from alibi.explainers import AnchorImage
except Exception as e:
    raise ImportError(
        "No puedo importar alibi.AnchorImage. Instala con: pip install alibi scikit-image"
    ) from e


/Users/haojie/PycharmProjects/TFM-Personalized-XAI/.venv/lib/python3.12/site-packages/alibi/explainers/shap_wrappers.py:302: SyntaxWarning: invalid escape sequence '\p'
  importance values to the model output. Since the feature importance values, :math:`\phi`, sum up to the
/Users/haojie/PycharmProjects/TFM-Personalized-XAI/.venv/lib/python3.12/site-packages/alibi/explainers/shap_wrappers.py:1043: SyntaxWarning: invalid escape sequence '\i'
  :math:`y \in \{0, 1\}`. Currently only binary cross-entropy and squared error losses can be explained. \


In [5]:
# Cargar modelo torchvision (ResNet50 preentrenado)

try:
    from torchvision.models import resnet50, ResNet50_Weights
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    imagenet_labels = weights.meta.get("categories", None)
except Exception:
    from torchvision.models import resnet50
    model = resnet50(pretrained=True)
    imagenet_labels = None

model = model.to(device).eval()
print("Modelo listo:", model.__class__.__name__)



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/haojie/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:04<00:00, 21.7MB/s]


Modelo listo: ResNet


In [6]:
# Utilidades: carga segura + preprocesado ImageNet (HWC [0,1] -> NCHW normalizado)

IM_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def safe_load_rgb(path: str, size=IM_SIZE):
    """Carga una imagen (segura) en RGB, la redimensiona y devuelve:
    - pil (RGB)
    - arr float32 en [0,1] con forma HxWx3
    """
    try:
        with Image.open(path) as im:
            im.verify()
        pil = Image.open(path).convert("RGB")
        pil = pil.resize((size, size))
        arr = np.asarray(pil).astype(np.float32) / 255.0
        return pil, arr
    except (UnidentifiedImageError, OSError) as e:
        print(f"Saltando archivo no válido: {path} | {e}")
        return None, None

def preprocess_batch(images_0_1: np.ndarray) -> torch.Tensor:
    """images_0_1: NxHxWx3 float32 [0,1] -> tensor Nx3xHxW normalizado."""
    x = (images_0_1 - MEAN) / STD
    x = torch.from_numpy(x).permute(0, 3, 1, 2).float()
    return x.to(device)

@torch.no_grad()
def predict_proba(images: np.ndarray) -> np.ndarray:
    """Para AnchorImage: recibe NxHxWx3 y devuelve probs NxC (numpy)."""
    if images.dtype != np.float32:
        images = images.astype(np.float32)
    x = preprocess_batch(images)
    logits = model(x)
    probs = F.softmax(logits, dim=1).detach().cpu().numpy()
    return probs

def top1_label(arr_0_1: np.ndarray):
    probs = predict_proba(arr_0_1[None, ...])[0]
    idx = int(np.argmax(probs))
    score = float(probs[idx])
    name = imagenet_labels[idx] if imagenet_labels is not None else str(idx)
    return idx, name, score


In [8]:
# CONFIG: carpeta de entrada y salida

INPUT_DIR = Path("../imagenes/original")    
OUTPUT_DIR = Path("../imagenes/output_anchor")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MAX_IMAGES = None  # None = todas

# Segmentación (slic / felzenszwalb / quickshift)
SEGMENTATION_FN = "slic"
SEGMENTATION_KWARGS = {
    "n_segments": 15,
    "compactness": 10,
    "sigma": 0.5,
    "start_label": 0
}

# Parámetros Anchor
P_SAMPLE = 0.5
THRESHOLD = 0.95
DELTA = 0.1
TAU = 0.15
BATCH_SIZE = 100
COVERAGE_SAMPLES = 2000   # si se quiere más estabilidad hay que aumentar
BEAM_SIZE = 1
STOP_ON_FIRST = False
SEED = 0

print("INPUT_DIR:", INPUT_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


INPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/original
OUTPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/output_anchor


In [9]:
# Inicializar explainer AnchorImage

image_shape = (IM_SIZE, IM_SIZE, 3)

explainer = AnchorImage(
    predictor=predict_proba,
    image_shape=image_shape,
    segmentation_fn=SEGMENTATION_FN,
    segmentation_kwargs=SEGMENTATION_KWARGS,
    seed=SEED
)

print("AnchorImage listo. Segmentation:", SEGMENTATION_FN, SEGMENTATION_KWARGS)


AnchorImage listo. Segmentation: slic {'n_segments': 15, 'compactness': 10, 'sigma': 0.5, 'start_label': 0}


In [10]:
# Ejecutar Anchors sobre todas las imágenes de la carpeta

paths = [p for p in sorted(INPUT_DIR.rglob("*")) if p.suffix.lower() in EXTS]
if MAX_IMAGES is not None:
    paths = paths[:MAX_IMAGES]

print("Imágenes encontradas:", len(paths))
if len(paths) == 0:
    raise FileNotFoundError(f"No encontré imágenes en {INPUT_DIR}. Revisa INPUT_DIR.")

processed = 0

for i, p in enumerate(paths, 1):
    pil, img_0_1 = safe_load_rgb(str(p), size=IM_SIZE)
    if pil is None:
        continue

    cls_idx, cls_name, cls_prob = top1_label(img_0_1)

    exp = explainer.explain(
        image=img_0_1,
        p_sample=P_SAMPLE,
        threshold=THRESHOLD,
        delta=DELTA,
        tau=TAU,
        batch_size=BATCH_SIZE,
        coverage_samples=COVERAGE_SAMPLES,
        beam_size=BEAM_SIZE,
        stop_on_first=STOP_ON_FIRST,
        verbose=False
    )

    anchor_img = getattr(exp, "anchor", None)
    segments = getattr(exp, "segments", None)
    precision = getattr(exp, "precision", None)
    coverage = getattr(exp, "coverage", None)

    if anchor_img is None:
        anchor_img = exp.data.get("anchor", None)
    if segments is None:
        segments = exp.data.get("segments", None)
    if precision is None:
        precision = exp.data.get("precision", None)
    if coverage is None:
        coverage = exp.data.get("coverage", None)

    raw = exp.data.get("raw", {}) if isinstance(exp.data, dict) else {}
    anchor_features = None
    if isinstance(raw, dict):
        anchor_features = raw.get("feature", None) or raw.get("features", None)

    # Construir máscara binaria
    mask = None
    if segments is not None and anchor_features is not None:
        mask = np.zeros_like(segments, dtype=np.uint8)
        for sp in anchor_features:
            # sp debería ser un int/np.int
            try:
                mask[segments == int(sp)] = 1
            except Exception:
                pass

    # Visualización: (1) original (2) anchor_img (3) segments (4) overlay con máscara si se puede
    out_png = OUTPUT_DIR / f"{p.stem}_anchor.png"

    fig = plt.figure(figsize=(12, 3))

    ax1 = plt.subplot(1, 4, 1)
    ax1.imshow(img_0_1)
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = plt.subplot(1, 4, 2)
    if anchor_img is not None:
        ax2.imshow(anchor_img)
        ax2.set_title("Anchor (masked image)")
    else:
        ax2.text(0.1, 0.5, "anchor_img=None", fontsize=10)
        ax2.set_title("Anchor")
    ax2.axis("off")

    ax3 = plt.subplot(1, 4, 3)
    if segments is not None:
        ax3.imshow(segments)
        ax3.set_title("Superpíxeles (segments)")
    else:
        ax3.text(0.1, 0.5, "segments=None", fontsize=10)
        ax3.set_title("Segments")
    ax3.axis("off")

    ax4 = plt.subplot(1, 4, 4)
    ax4.imshow(img_0_1)
    if mask is not None:
        ax4.imshow(mask, cmap="jet", alpha=0.45)
        ax4.set_title(f"Overlay\n{cls_name} ({cls_prob:.2f})\nprec={precision:.2f}, cov={coverage:.2f}")
    else:
        ax4.set_title(f"Overlay (sin máscara)\n{cls_name} ({cls_prob:.2f})\nprec={precision}, cov={coverage}")
    ax4.axis("off")

    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)

    # Guardar resumen TXT
    out_txt = OUTPUT_DIR / f"{p.stem}_anchor.txt"
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(f"file: {p}\n")
        f.write(f"pred_class_idx: {cls_idx}\n")
        f.write(f"pred_class_name: {cls_name}\n")
        f.write(f"pred_prob: {cls_prob:.6f}\n")
        f.write(f"precision: {precision}\n")
        f.write(f"coverage: {coverage}\n")
        f.write(f"anchor_features (superpixels): {anchor_features}\n")

    processed += 1
    if processed % 3 == 0 or i == len(paths):
        print(f"Procesadas: {processed} | Último guardado: {out_png}")

print("Revisa la carpeta output_anchor/")


Imágenes encontradas: 11
Procesadas: 3 | Último guardado: ../imagenes/output_anchor/image03_anchor.png
Procesadas: 6 | Último guardado: ../imagenes/output_anchor/image06_anchor.png
Procesadas: 9 | Último guardado: ../imagenes/output_anchor/image09_anchor.png
Procesadas: 11 | Último guardado: ../imagenes/output_anchor/image11_anchor.png
Revisa la carpeta output_anchor/
